In [1]:
%pip install -U langchain langchain-core langchain-google-genai langchain-community sqlalchemy "requests==2.32.4"

  Using cached langchain-1.2.12-py3-none-any.whl.metadata (5.7 kB)
  Using cached langchain_core-1.2.19-py3-none-any.whl.metadata (4.4 kB)
  Using cached langchain_google_genai-4.2.1-py3-none-any.whl.metadata (2.7 kB)
  Using cached langchain_community-0.4.1-py3-none-any.whl.metadata (3.0 kB)
  Using cached langchain_classic-1.0.3-py3-none-any.whl.metadata (4.8 kB)
INFO: pip is looking at multiple versions of langchain-community to determine which version is compatible with other requirements. This could take a while.
  Using cached langchain_community-0.4-py3-none-any.whl.metadata (3.0 kB)
  Using cached langchain_community-0.3.31-py3-none-any.whl.metadata (3.0 kB)
  Using cached langchain_community-0.3.30-py3-none-any.whl.metadata (3.0 kB)
  Using cached langchain_community-0.3.29-py3-none-any.whl.metadata (2.9 kB)
  Using cached langchain_community-0.3.28-py3-none-any.whl.metadata (2.9 kB)
  Using cached langchain_community-0.3.27-py3-none-any.whl.metadata (2.9 kB)
INFO: pip is stil

Imports & Setup

In [3]:
import os
import sqlite3
from typing import Optional
from sqlalchemy import create_engine, text
from langchain_community.utilities import SQLDatabase
from langchain.chains import create_sql_query_chain
from langchain_google_genai import ChatGoogleGenerativeAI


def load_google_api_key() -> str:
    for key_name in ("GOOGLE_API_KEY", "GEMINI_API_KEY"):
        value = os.getenv(key_name)
        if value:
            os.environ["GOOGLE_API_KEY"] = value
            os.environ["GEMINI_API_KEY"] = value
            return value
    try:
        from google.colab import userdata
        for key_name in ("GOOGLE_API_KEY", "GEMINI_API_KEY"):
            try:
                value = userdata.get(key_name)
                if value:
                    os.environ["GOOGLE_API_KEY"] = value
                    os.environ["GEMINI_API_KEY"] = value
                    return value
            except Exception:
                pass
    except Exception:
        pass
    raise RuntimeError("Gemini API key not found.")


def message_to_text(message) -> str:
    text_attr = getattr(message, "text", None)
    if isinstance(text_attr, str) and text_attr.strip():
        return text_attr.strip()
    content = getattr(message, "content", None)
    if isinstance(content, str):
        return content.strip()
    if isinstance(content, list):
        parts = []
        for item in content:
            if isinstance(item, str):
                parts.append(item)
            elif isinstance(item, dict):
                t = item.get("text")
                if t:
                    parts.append(t)
        return "\n".join(parts).strip()
    return ""


API_KEY = load_google_api_key()
MODEL_NAME = "gemini-2.5-flash"

llm = ChatGoogleGenerativeAI(
    model=MODEL_NAME,
    temperature=0,
)

print(f"Loaded API key. Using model: {MODEL_NAME}")

Loaded API key. Using model: gemini-2.5-flash


Create SQLite Database with Structured Data

In [4]:
DB_PATH = "/content/structured_data.db"

conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

# Clean reruns
cursor.execute("DROP TABLE IF EXISTS scores")
cursor.execute("DROP TABLE IF EXISTS attendance")
cursor.execute("DROP TABLE IF EXISTS students")

# Students table
cursor.execute("""
CREATE TABLE students (
    student_id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    department TEXT NOT NULL,
    year INTEGER NOT NULL
)
""")

# Scores table
cursor.execute("""
CREATE TABLE scores (
    score_id INTEGER PRIMARY KEY,
    student_id INTEGER,
    subject TEXT NOT NULL,
    marks INTEGER NOT NULL,
    FOREIGN KEY (student_id) REFERENCES students(student_id)
)
""")

# Attendance table
cursor.execute("""
CREATE TABLE attendance (
    attendance_id INTEGER PRIMARY KEY,
    student_id INTEGER,
    subject TEXT NOT NULL,
    attended INTEGER NOT NULL,
    total INTEGER NOT NULL,
    FOREIGN KEY (student_id) REFERENCES students(student_id)
)
""")

students_data = [
    (1, "Arjun Sharma",   "Computer Science", 2),
    (2, "Priya Patel",    "Computer Science", 2),
    (3, "Rahul Das",      "Electronics",      3),
    (4, "Sneha Roy",      "Computer Science", 1),
    (5, "Vikram Singh",   "Electronics",      2),
    (6, "Ananya Ghosh",   "Mathematics",      3),
    (7, "Rohan Mehta",    "Computer Science", 3),
    (8, "Diya Kapoor",    "Mathematics",      1),
]
cursor.executemany("INSERT INTO students VALUES (?, ?, ?, ?)", students_data)

scores_data = [
    (1,  1, "Mathematics",        88),
    (2,  1, "Python",             92),
    (3,  1, "AI",                 85),
    (4,  2, "Mathematics",        95),
    (5,  2, "Python",             89),
    (6,  2, "AI",                 91),
    (7,  3, "Mathematics",        72),
    (8,  3, "Digital Electronics",80),
    (9,  3, "AI",                 76),
    (10, 4, "Mathematics",        65),
    (11, 4, "Python",             70),
    (12, 4, "AI",                 68),
    (13, 5, "Mathematics",        78),
    (14, 5, "Digital Electronics",82),
    (15, 5, "AI",                 79),
    (16, 6, "Mathematics",        97),
    (17, 6, "Statistics",         94),
    (18, 6, "AI",                 88),
    (19, 7, "Mathematics",        83),
    (20, 7, "Python",             87),
    (21, 7, "AI",                 90),
    (22, 8, "Mathematics",        60),
    (23, 8, "Statistics",         63),
    (24, 8, "AI",                 58),
]
cursor.executemany("INSERT INTO scores VALUES (?, ?, ?, ?)", scores_data)

attendance_data = [
    (1,  1, "Mathematics",        28, 30),
    (2,  1, "Python",             25, 30),
    (3,  2, "Mathematics",        30, 30),
    (4,  2, "Python",             29, 30),
    (5,  3, "Mathematics",        20, 30),
    (6,  3, "Digital Electronics",22, 30),
    (7,  4, "Mathematics",        18, 30),
    (8,  4, "Python",             15, 30),
    (9,  5, "Mathematics",        27, 30),
    (10, 5, "Digital Electronics",28, 30),
    (11, 6, "Mathematics",        30, 30),
    (12, 6, "Statistics",         29, 30),
    (13, 7, "Mathematics",        24, 30),
    (14, 7, "Python",             26, 30),
    (15, 8, "Mathematics",        12, 30),
    (16, 8, "Statistics",         14, 30),
]
cursor.executemany("INSERT INTO attendance VALUES (?, ?, ?, ?, ?)", attendance_data)

conn.commit()
conn.close()

print(f"SQLite database created at: {DB_PATH}")
print("Tables: students, scores, attendance")

SQLite database created at: /content/structured_data.db
Tables: students, scores, attendance


Why RAG Fails on Structured Data

In [5]:
print("=" * 80)
print("DEMONSTRATING WHY RAG FAILS ON STRUCTURED DATA")
print("=" * 80)

# Simulate RAG: rows converted to text chunks stored in a vector store
sample_rows_as_text = """
student_id=1, name=Arjun Sharma,  department=Computer Science, subject=Mathematics, marks=88
student_id=2, name=Priya Patel,   department=Computer Science, subject=Mathematics, marks=95
student_id=3, name=Rahul Das,     department=Electronics,      subject=Mathematics, marks=72
student_id=4, name=Sneha Roy,     department=Computer Science, subject=Mathematics, marks=65
student_id=5, name=Vikram Singh,  department=Electronics,      subject=Mathematics, marks=78
student_id=6, name=Ananya Ghosh,  department=Mathematics,      subject=Mathematics, marks=97
student_id=7, name=Rohan Mehta,   department=Computer Science, subject=Mathematics, marks=83
student_id=8, name=Diya Kapoor,   department=Mathematics,      subject=Mathematics, marks=60
"""

rag_fail_prompt = f"""
You are a RAG system. You retrieved the following text chunks from a vector store.
Answer ONLY from these chunks. Do NOT compute anything yourself.

Retrieved chunks:
{sample_rows_as_text}

Question: What is the average Mathematics score of all Computer Science students?
""".strip()

rag_response = llm.invoke(rag_fail_prompt)
print("\n[RAG ATTEMPT]")
print("Question: What is the average Mathematics score of all Computer Science students?")
print("\nRAG Answer:")
print(message_to_text(rag_response))

# Now show the correct answer via direct SQL
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()
cursor.execute("""
    SELECT AVG(sc.marks)
    FROM scores sc
    JOIN students st ON sc.student_id = st.student_id
    WHERE sc.subject = 'Mathematics' AND st.department = 'Computer Science'
""")
correct = cursor.fetchone()[0]
conn.close()

print(f"\n[CORRECT ANSWER via SQL]: {correct:.2f}")
print()
print("CONCLUSION:")
print("RAG cannot aggregate, filter by exact values, or join tables reliably.")
print("It guesses from text. Text-to-SQL executes precise queries — exact answers every time.")

DEMONSTRATING WHY RAG FAILS ON STRUCTURED DATA

[RAG ATTEMPT]
Question: What is the average Mathematics score of all Computer Science students?

RAG Answer:
The average Mathematics score of all Computer Science students cannot be determined from the provided chunks as computation is not allowed. The chunks provide individual student scores, but not pre-computed averages.

[CORRECT ANSWER via SQL]: 82.75

CONCLUSION:
RAG cannot aggregate, filter by exact values, or join tables reliably.
It guesses from text. Text-to-SQL executes precise queries — exact answers every time.


Build the Text-to-SQL Pipeline

In [9]:
# # LangChain SQLDatabase wrapper auto-reads the schema
# db = SQLDatabase.from_uri(f"sqlite:///{DB_PATH}")

# print("Tables found:", db.get_usable_table_names())
# print()
# print("Schema preview:")
# print(db.get_table_info())

# # create_sql_query_chain: LLM reads schema + question → generates SQL
# sql_chain = create_sql_query_chain(llm, db)


# def run_text_to_sql(question: str) -> dict:
#     """
#     Full Text-to-SQL pipeline:
#     Step 1 — LLM generates SQL from the natural language question
#     Step 2 — SQL is executed on the SQLite database
#     Step 3 — LLM interprets raw DB result into a natural language answer
#     """

#     # Step 1: Generate SQL
#     generated_sql = sql_chain.invoke({"question": question})

#     # Clean markdown fences if LLM wraps SQL in ```sql ... ```
#     clean_sql = generated_sql.strip()
#     if clean_sql.startswith("```"):
#         lines = clean_sql.split("\n")
#         clean_sql = "\n".join(lines[1:-1]).strip()

#     # Step 2: Execute SQL on the database
#     try:
#         engine = create_engine(f"sqlite:///{DB_PATH}")
#         with engine.connect() as connection:
#             result = connection.execute(text(clean_sql))
#             rows = result.fetchall()
#             columns = list(result.keys())
#     except Exception as e:
#         return {
#             "question": question,
#             "generated_sql": clean_sql,
#             "raw_result": None,
#             "final_answer": f"SQL execution error: {e}",
#         }

#     # Step 3: Format raw result as a readable table
#     if not rows:
#         raw_result_text = "No results found."
#     else:
#         header = " | ".join(columns)
#         divider = "-" * len(header)
#         row_lines = [" | ".join(str(v) for v in row) for row in rows]
#         raw_result_text = "\n".join([header, divider] + row_lines)

#     # Step 4: LLM interprets the raw result into plain English
#     interpretation_prompt = f"""
# You are a helpful data assistant for a student database.
# Convert the SQL query result below into a clear, concise natural language answer in 1-3 sentences.

# User question: {question}
# SQL query used: {clean_sql}
# Query result:
# {raw_result_text}
# """.strip()

#     final_response = llm.invoke(interpretation_prompt)

#     return {
#         "question": question,
#         "generated_sql": clean_sql,
#         "raw_result": raw_result_text,
#         "final_answer": message_to_text(final_response),
#     }


# def print_result(result: dict) -> None:
#     print("=" * 80)
#     print(f"Question     : {result['question']}")
#     print(f"Generated SQL: {result['generated_sql']}")
#     print(f"Raw Result   :\n{result['raw_result']}")
#     print(f"Final Answer : {result['final_answer']}")
#     print()



# LangChain SQLDatabase wrapper auto-reads the schema
db = SQLDatabase.from_uri(f"sqlite:///{DB_PATH}")

print("Tables found:", db.get_usable_table_names())
print()
print("Schema preview:")
print(db.get_table_info())

# create_sql_query_chain: LLM reads schema + question → generates SQL
sql_chain = create_sql_query_chain(llm, db)


def clean_generated_sql(raw: str) -> str:
    """Strip all known prefixes and wrappers Gemini might add around SQL."""
    sql = raw.strip()

    # Remove markdown fences: ```sql ... ``` or ``` ... ```
    if sql.startswith("```"):
        lines = sql.split("\n")
        # Drop first line (```sql or ```) and last line (```)
        sql = "\n".join(lines[1:-1]).strip()

    # Remove "SQLQuery:" prefix (LangChain default prompt style)
    if sql.lower().startswith("sqlquery:"):
        sql = sql[len("sqlquery:"):].strip()

    # Remove "sql" if it appears alone on the first line
    if sql.lower().startswith("sql\n"):
        sql = sql[4:].strip()

    # Remove any leading/trailing whitespace again after all stripping
    return sql.strip()


def run_text_to_sql(question: str) -> dict:
    """
    Full Text-to-SQL pipeline:
    Step 1 — LLM generates SQL from the natural language question
    Step 2 — Clean and strip any unwanted prefix from the SQL
    Step 3 — Execute the SQL on the SQLite database
    Step 4 — LLM interprets the raw result into a natural language answer
    """

    # Step 1: Generate SQL
    generated_sql = sql_chain.invoke({"question": question})

    # Step 2: Clean the SQL
    clean_sql = clean_generated_sql(generated_sql)

    # Step 3: Execute SQL on the database
    try:
        engine = create_engine(f"sqlite:///{DB_PATH}")
        with engine.connect() as connection:
            result = connection.execute(text(clean_sql))
            rows = result.fetchall()
            columns = list(result.keys())
    except Exception as e:
        return {
            "question":      question,
            "generated_sql": clean_sql,
            "raw_result":    None,
            "final_answer":  f"SQL execution error: {e}",
        }

    # Step 4: Format raw result as a readable table
    if not rows:
        raw_result_text = "No results found."
    else:
        header    = " | ".join(columns)
        divider   = "-" * len(header)
        row_lines = [" | ".join(str(v) for v in row) for row in rows]
        raw_result_text = "\n".join([header, divider] + row_lines)

    # Step 5: LLM interprets the raw result into plain English
    interpretation_prompt = f"""
You are a helpful data assistant for a student database.
Convert the SQL query result below into a clear, concise natural language answer in 1-3 sentences.

User question: {question}
SQL query used: {clean_sql}
Query result:
{raw_result_text}
""".strip()

    final_response = llm.invoke(interpretation_prompt)

    return {
        "question":      question,
        "generated_sql": clean_sql,
        "raw_result":    raw_result_text,
        "final_answer":  message_to_text(final_response),
    }


def print_result(result: dict) -> None:
    print("=" * 80)
    print(f"Question     : {result['question']}")
    print(f"Generated SQL: {result['generated_sql']}")
    print(f"Raw Result   :\n{result['raw_result']}")
    print(f"Final Answer : {result['final_answer']}")
    print()

print("Text-to-SQL pipeline ready.")


Tables found: ['attendance', 'scores', 'students']

Schema preview:

CREATE TABLE attendance (
	attendance_id INTEGER, 
	student_id INTEGER, 
	subject TEXT NOT NULL, 
	attended INTEGER NOT NULL, 
	total INTEGER NOT NULL, 
	PRIMARY KEY (attendance_id), 
	FOREIGN KEY(student_id) REFERENCES students (student_id)
)

/*
3 rows from attendance table:
attendance_id	student_id	subject	attended	total
1	1	Mathematics	28	30
2	1	Python	25	30
3	2	Mathematics	30	30
*/


CREATE TABLE scores (
	score_id INTEGER, 
	student_id INTEGER, 
	subject TEXT NOT NULL, 
	marks INTEGER NOT NULL, 
	PRIMARY KEY (score_id), 
	FOREIGN KEY(student_id) REFERENCES students (student_id)
)

/*
3 rows from scores table:
score_id	student_id	subject	marks
1	1	Mathematics	88
2	1	Python	92
3	1	AI	85
*/


CREATE TABLE students (
	student_id INTEGER, 
	name TEXT NOT NULL, 
	department TEXT NOT NULL, 
	year INTEGER NOT NULL, 
	PRIMARY KEY (student_id)
)

/*
3 rows from students table:
student_id	name	department	year
1	Arjun Sharm

In [10]:
import time

demo_questions = [
    "Who scored the highest marks in Mathematics?",
    "What is the average score of Computer Science students across all subjects?",
    "Which students have attendance below 70% in any subject?",
    "List all students in year 3 along with their department.",
    "Which subject has the highest average marks overall?",
    "How many students are there in each department?",
]

for question in demo_questions:
    result = run_text_to_sql(question)
    print_result(result)
    time.sleep(15)  # Respect free tier: 5 requests per minute

Question     : Who scored the highest marks in Mathematics?
Generated SQL: SELECT T1.name FROM students AS T1 INNER JOIN scores AS T2 ON T1.student_id = T2.student_id WHERE T2.subject = 'Mathematics' ORDER BY T2.marks DESC LIMIT 1
Raw Result   :
name
----
Ananya Ghosh
Final Answer : Ananya Ghosh scored the highest marks in Mathematics.

Question     : What is the average score of Computer Science students across all subjects?
Generated SQL: SELECT AVG(T2.marks) FROM students AS T1 INNER JOIN scores AS T2 ON T1.student_id = T2.student_id WHERE T1.department = 'Computer Science'
Raw Result   :
AVG(T2.marks)
-------------
83.58333333333333
Final Answer : The average score for Computer Science students across all subjects is approximately 83.58.

Question     : Which students have attendance below 70% in any subject?
Generated SQL: SELECT
  T1.name,
  T2.subject,
  CAST(T2.attended AS REAL) * 100 / T2.total AS attendance_percentage
FROM students AS T1
INNER JOIN attendance AS T2
  ON T1.st

 Interactive Mode

In [11]:
print("=" * 80)
print("INTERACTIVE MODE — Ask anything about the student database")
print("Type 'exit' to quit.")
print("=" * 80)

while True:
    user_question = input("\nYour question: ").strip()
    if user_question.lower() in ("exit", "quit", ""):
        print("Exiting.")
        break
    result = run_text_to_sql(user_question)
    print_result(result)

INTERACTIVE MODE — Ask anything about the student database
Type 'exit' to quit.

Your question: Highest marks in Electronics
Question     : Highest marks in Electronics
Generated SQL: SELECT MAX(T2.marks) FROM students AS T1 INNER JOIN scores AS T2 ON T1.student_id = T2.student_id WHERE T1.department = 'Electronics'
Raw Result   :
MAX(T2.marks)
-------------
82
Final Answer : The highest mark achieved by a student in the Electronics department is 82.


Your question: exit
Exiting.
